# Download Jina artifacts from Azure (CPU only)

This Colab notebook uses no GPU. It downloads the Jina mapping needed for OCR/ASR remapping from Azure Blob Storage to Google Drive. Azure credentials stay in the Colab secret `AZURE_STORAGE_CONNECTION_STRING`.

In [ ]:
!pip -q install azure-storage-blob tqdm

In [ ]:
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
AZURE_STORAGE_CONNECTION_STRING = userdata.get('AZURE_STORAGE_CONNECTION_STRING') or os.environ.get('AZURE_STORAGE_CONNECTION_STRING', '')
if not AZURE_STORAGE_CONNECTION_STRING:
    raise RuntimeError('Create the Colab secret AZURE_STORAGE_CONNECTION_STRING, then rerun this cell.')

EMBEDDINGS_CONTAINER = 'embeddings'
EMBEDDING_RUN = 'fine_keyframes_jina_clip_v2_1024d_v2'
DOWNLOAD_ALL_JINA_ARTIFACTS = False  # False downloads only global_ids.parquet for OCR/ASR remapping.
DRIVE_DIR = Path('/content/drive/MyDrive/AIC_2026/jina_artifacts') / EMBEDDING_RUN
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print('Destination:', DRIVE_DIR)

In [ ]:
from azure.storage.blob import BlobServiceClient
from tqdm.auto import tqdm

artifact_names = ['global_ids.parquet']
if DOWNLOAD_ALL_JINA_ARTIFACTS:
    artifact_names = ['global_ids.parquet', 'video_metadata.parquet', 'jina_faiss.index', 'index_meta.json']

container = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING).get_container_client(EMBEDDINGS_CONTAINER)
for name in artifact_names:
    remote = f'indexes/{EMBEDDING_RUN}/jina/{name}'
    local = DRIVE_DIR / name
    properties = container.get_blob_client(remote).get_blob_properties()
    expected_size = properties.size
    if local.exists() and local.stat().st_size == expected_size:
        print(f'Already downloaded: {local.name} ({expected_size:,} bytes)')
        continue

    temporary = local.with_suffix(local.suffix + '.part')
    with temporary.open('wb') as handle, tqdm(total=expected_size, unit='B', unit_scale=True, desc=name) as bar:
        stream = container.get_blob_client(remote).download_blob(max_concurrency=4)
        for chunk in stream.chunks():
            handle.write(chunk)
            bar.update(len(chunk))
    if temporary.stat().st_size != expected_size:
        raise RuntimeError(f'Incomplete download for {name}: {temporary.stat().st_size}/{expected_size} bytes')
    temporary.replace(local)
    print(f'Downloaded: {local} ({expected_size:,} bytes)')

## Next step

For remapping only, download `global_ids.parquet` from Google Drive to the backend, then run `remap_ocr_to_jina.py` and `remap_asr_to_jina.py`. Set `DOWNLOAD_ALL_JINA_ARTIFACTS = True` only when the backend also needs the FAISS index and metadata artifacts.